# Student Notebook 01 — Modeling (Layer 1)

Pick a model, tune it, evaluate it. Every teammate runs this
notebook with a different choice in the CONFIG cell below
and shares the headline AUC number.

**Run order:** this is notebook 01. After running it, run
02 → 03 → 04, and only then 05 (the test set).

**What you'll get out:** CV AUC (mean ± std) from repeated
cross-validation, held-out cal AUC, a tuned model saved at
``data/processed/student/<run_name>/student_model.joblib``
for the next notebooks.

**What's bundled in this notebook (vs the bare baseline):**

1. **Hyperparameter search** via ``GridSearchCV`` over a small
   grid per model family. Often the biggest single lift.
2. **Repeated stratified cross-validation** (5 folds × 3
   repetitions = 15 fold values) for tighter CV-AUC
   confidence intervals.
3. **Class balancing** (``class_weight="balanced"`` for sklearn
   models, ``scale_pos_weight`` for XGBoost) to compensate
   for the 60/40 class imbalance on roi_gt_2.

Each of the three is toggleable in CONFIG so you can ablate
and see what each one buys you.

## CONFIG — edit me

Change any value below and re-run the notebook. The
comments list the supported options.

In [ ]:
CONFIG = {
    # YOUR run_name. Use a unique label so your artifacts don't
    # clobber teammates'. Convention: "<your-name>_<model>".
    # Each run lands in data/processed/student/<run_name>/.
    "run_name": "team_baseline_rf",

    # Target. The headline target is roi_gt_2 (was the film
    # net profitable?). roi_gt_1 is the easier 1x ROI binary;
    # log_roi is regression. The other two notebooks assume
    # classification, so for the team comparison stick with
    # roi_gt_2 unless you want to dig into log_roi separately.
    "target": "roi_gt_2",                # "roi_gt_2" | "roi_gt_1" | "log_roi"

    # Model family. Default = random_forest because it wins
    # the bake-off on the simplified 127-feature matrix:
    #   random_forest  : CV AUC 0.595, cal AUC 0.612  <-- best in general
    #   xgboost        : CV AUC 0.597, cal AUC 0.569  (CV tied, weaker on cal)
    #   logistic       : CV AUC 0.576, cal AUC 0.573
    #   svm_rbf        : CV AUC 0.571, cal AUC 0.530
    # Try a different family to see how the choice moves the needle.
    "model_family": "random_forest",      # "logistic" | "random_forest" | "xgboost" | "svm_rbf"

    # Feature subset. "all" uses the 127-feature matrix from
    # Phase 3. The others let you isolate one feature group
    # to see how much each one matters.
    "feature_set": "all",                 # "all" | "structural" | "topic" | "embedding" | "network"

    # ---- Improvement toggles (all True by default) ----
    # Flip any of these to False to ablate and see how much
    # each one contributes to the headline AUC.
    "use_grid_search":   True,            # GridSearchCV on a small per-family grid
    "use_repeated_cv":   True,            # 5 folds x n_repeats stratified CV
    "use_class_balance": True,            # class_weight='balanced' / scale_pos_weight

    # CV / random seed. Don't change unless you know why.
    "n_splits":   5,
    "n_repeats":  3,                      # used when use_repeated_cv=True
    "random_seed": 42,
}

## Imports and paths

In [ ]:
# --- Paths (works from any working directory) ---
from pathlib import Path

def _find_project_root() -> Path:
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / "docs" / "PROJECT_CONTEXT.md").is_file():
            return cand
    raise RuntimeError(f"Could not locate project root from {Path.cwd()!s}")

PROJECT_ROOT = _find_project_root()
DATA = PROJECT_ROOT / "data" / "processed"
STUDENT = DATA / "student" / CONFIG["run_name"]
STUDENT.mkdir(parents=True, exist_ok=True)
print(f"Project root:  {PROJECT_ROOT}")
print(f"Run name:      {CONFIG['run_name']!r}")
print(f"Artifacts go:  {STUDENT.relative_to(PROJECT_ROOT)}")

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import (
    GridSearchCV,
    RepeatedStratifiedKFold,
    StratifiedKFold,
    cross_val_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

## Load features and targets

We use the 92-feature matrix that the rigorous pipeline
settled on (``standalone_positive_union_mpnet`` from
Phase 3). Train + cal splits only — the test split is for
notebook 05.

In [ ]:
# Read the prebuilt feature matrix. imdb_id is its index;
# ``split`` is already a column ('train' / 'cal' / 'test').
feat = pd.read_parquet(DATA / "features.parquet").reset_index()
print(f"Total films in the matrix: {len(feat)}")
print(f"Split breakdown: {feat['split'].value_counts().to_dict()}")

# Train + cal pool (the test split is held out for notebook 05).
df = feat[feat["split"].isin(["train", "cal"])].reset_index(drop=True)
print(f"Films available in this notebook: {len(df)}")

# Pick feature columns based on CONFIG['feature_set'].
non_feat = {"imdb_id", "split", "log_roi", "roi_gt_1", "roi_gt_2"}
all_cols = [c for c in df.columns if c not in non_feat]
groups = {
    "all":        all_cols,
    "structural": [c for c in all_cols if c.startswith(("log_", "n_", "dialogue_to_", "release_year", "log_runtime", "genre_"))],
    "topic":      [c for c in all_cols if c.startswith("topic_")],
    "embedding":  [c for c in all_cols if c.startswith("embed_pc_")],
    "network":    [c for c in all_cols if c.startswith("network_")],
}
feat_cols = groups[CONFIG["feature_set"]]
print(f"Feature set: {CONFIG['feature_set']!r} -> {len(feat_cols)} columns")

X = df[feat_cols].fillna(0).values
y = df[CONFIG["target"]].astype(int).values if CONFIG["target"].startswith("roi_gt") else df[CONFIG["target"]].values
print(f"X shape: {X.shape}, y positive rate: {y.mean():.3f}")

## Build the model — with optional class balancing

Each model family is wrapped in a sklearn ``Pipeline`` with
a ``StandardScaler`` so the input distribution is consistent.
SVM also goes through ``probability=True`` so the next
notebook can calibrate it.

When ``CONFIG["use_class_balance"]`` is True we apply:

- ``class_weight="balanced"`` for logistic / random forest /
  SVM (sklearn's standard trick: weights inversely
  proportional to class frequency).
- ``scale_pos_weight = n_neg / n_pos`` for XGBoost (its
  equivalent parameter).

The 60/40 class imbalance on roi_gt_2 is mild, so this is a
small tweak — but free.

In [ ]:
# Compute the imbalance ratio for XGBoost upfront.
train_mask = (df["split"] == "train").values
cal_mask   = (df["split"] == "cal").values
y_train = y[train_mask]
y_cal   = y[cal_mask]
n_neg = int((y_train == 0).sum())
n_pos = int((y_train == 1).sum())
scale_pos_weight = n_neg / max(n_pos, 1)
print(f"Train class balance: {n_neg} negative, {n_pos} positive  ->  scale_pos_weight = {scale_pos_weight:.3f}")

def build_model(family: str, seed: int = 42, balance: bool = False):
    cw = "balanced" if balance else None
    if family == "logistic":
        clf = LogisticRegression(max_iter=2000, C=1.0, class_weight=cw, random_state=seed)
    elif family == "random_forest":
        clf = RandomForestClassifier(
            n_estimators=300, max_depth=None, min_samples_leaf=2,
            class_weight=cw, random_state=seed, n_jobs=-1,
        )
    elif family == "xgboost":
        if not HAS_XGB:
            raise RuntimeError("xgboost not installed. pip install xgboost")
        spw = scale_pos_weight if balance else 1.0
        clf = XGBClassifier(
            n_estimators=300, max_depth=4, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            scale_pos_weight=spw,
            eval_metric="logloss", random_state=seed, n_jobs=-1,
        )
    elif family == "svm_rbf":
        clf = SVC(kernel="rbf", C=1.0, gamma="scale", probability=True,
                  class_weight=cw, random_state=seed)
    else:
        raise ValueError(f"Unknown model_family: {family!r}")
    return Pipeline([("scaler", StandardScaler(with_mean=False)), ("model", clf)])

base_model = build_model(
    CONFIG["model_family"],
    seed=CONFIG["random_seed"],
    balance=CONFIG["use_class_balance"],
)
print(f"Built {CONFIG['model_family']!r} model | "
      f"class_balance={CONFIG['use_class_balance']}")

## Cross-validation scheme — repeated or single?

- ``StratifiedKFold(n_splits=5)``: 5 fold values per CV run.
  Cheap but noisy on a 1456-film pool.
- ``RepeatedStratifiedKFold(n_splits=5, n_repeats=3)``: 15
  fold values from three independent shuffles. Same mean,
  tighter CI.

``CONFIG["use_repeated_cv"]`` toggles between them.

In [ ]:
if CONFIG["use_repeated_cv"]:
    cv = RepeatedStratifiedKFold(
        n_splits=CONFIG["n_splits"],
        n_repeats=CONFIG["n_repeats"],
        random_state=CONFIG["random_seed"],
    )
    cv_label = f"RepeatedStratifiedKFold({CONFIG['n_splits']} folds x {CONFIG['n_repeats']} repeats = {CONFIG['n_splits']*CONFIG['n_repeats']} fold values)"
else:
    cv = StratifiedKFold(
        n_splits=CONFIG["n_splits"], shuffle=True,
        random_state=CONFIG["random_seed"],
    )
    cv_label = f"StratifiedKFold({CONFIG['n_splits']} fold values)"
print(f"CV: {cv_label}")

## Tune hyperparameters with ``GridSearchCV``

Default hyperparameters leave 3-5pp of AUC on the table for
tree models. The grid below is small (3-12 candidates per
family) so a 15-fold repeated CV finishes in a couple of
minutes.

When ``CONFIG["use_grid_search"]`` is False, we skip the
search and run plain ``cross_val_score`` on the default
hyperparameters.

In [ ]:
# Per-family small grids. Keep them small so the run finishes
# in 1-3 minutes on a laptop.
GRIDS = {
    "logistic": {
        "model__C": [0.1, 1.0, 10.0],
    },
    "random_forest": {
        "model__n_estimators":     [200, 500],
        "model__max_depth":        [None, 10, 20],
        "model__min_samples_leaf": [1, 5, 10],
    },
    "xgboost": {
        "model__max_depth":     [3, 4, 6],
        "model__learning_rate": [0.05, 0.1],
        "model__n_estimators":  [200, 500],
    },
    "svm_rbf": {
        "model__C":     [0.5, 1.0, 3.0],
        "model__gamma": ["scale", 0.01, 0.1],
    },
}

if CONFIG["use_grid_search"]:
    grid = GRIDS[CONFIG["model_family"]]
    n_combos = int(np.prod([len(v) for v in grid.values()]))
    print(f"Searching {n_combos} hyperparameter combinations on {cv_label}")
    search = GridSearchCV(base_model, grid, cv=cv, scoring="roc_auc",
                           n_jobs=-1, refit=True)
    search.fit(X[train_mask | cal_mask], y[train_mask | cal_mask])
    cv_aucs = search.cv_results_["mean_test_score"]
    cv_auc_mean = float(search.best_score_)
    cv_auc_std  = float(search.cv_results_["std_test_score"][search.best_index_])
    best_params = search.best_params_
    model = search.best_estimator_
    print(f"\nBest params: {best_params}")
    print(f"Best CV AUC: {cv_auc_mean:.3f} +/- {cv_auc_std:.3f}")
else:
    scores = cross_val_score(base_model, X[train_mask | cal_mask], y[train_mask | cal_mask],
                              cv=cv, scoring="roc_auc", n_jobs=-1)
    cv_auc_mean = float(scores.mean())
    cv_auc_std  = float(scores.std())
    best_params = "(no grid search; defaults)"
    model = base_model
    print(f"CV AUC: {cv_auc_mean:.3f} +/- {cv_auc_std:.3f}")

# Bootstrap CI on the CV-fold AUC distribution (same as before).
if CONFIG["use_grid_search"]:
    # Use the per-fold AUCs from the BEST hyperparameter cell.
    best_idx = search.best_index_
    fold_keys = [k for k in search.cv_results_ if k.startswith("split") and k.endswith("_test_score")]
    best_fold_aucs = np.array([search.cv_results_[k][best_idx] for k in fold_keys])
else:
    best_fold_aucs = scores
rng = np.random.default_rng(CONFIG["random_seed"])
boot = np.array([rng.choice(best_fold_aucs, size=len(best_fold_aucs), replace=True).mean()
                  for _ in range(2000)])
ci_lo, ci_hi = np.quantile(boot, [0.025, 0.975])
print(f"95% bootstrap CI on the CV mean: [{ci_lo:.3f}, {ci_hi:.3f}]")

## Held-out evaluation on the cal split

With ``GridSearchCV(refit=True)`` the search already refit
the best model on all 1,456 train+cal films, so the cal AUC
below is **in-sample** for the tuned model. With
``use_grid_search=False`` it's still genuinely held out
(we explicitly fit on train and score on cal). We report
both interpretations so it's clear what each cell shows.

In [ ]:
X_train_only, _y_train = X[train_mask], y[train_mask]
X_cal,        _y_cal   = X[cal_mask],   y[cal_mask]

if CONFIG["use_grid_search"]:
    # The model is already refit on train+cal; cal AUC is in-sample.
    cal_proba = model.predict_proba(X_cal)[:, 1]
    cal_auc = roc_auc_score(_y_cal, cal_proba)
    print(f"Cal-set AUC (in-sample, refit on train+cal): {cal_auc:.3f}")

    # Also fit a fresh copy on TRAIN ONLY for the genuinely held-out figure.
    from sklearn.base import clone
    held_out = clone(model).fit(X_train_only, _y_train)
    held_proba = held_out.predict_proba(X_cal)[:, 1]
    held_auc = roc_auc_score(_y_cal, held_proba)
    print(f"Cal-set AUC (held out, fit on train only):  {held_auc:.3f}")
    cal_auc = held_auc  # save the honest one downstream
else:
    model.fit(X_train_only, _y_train)
    cal_proba = model.predict_proba(X_cal)[:, 1]
    cal_auc = roc_auc_score(_y_cal, cal_proba)
    print(f"Cal-set AUC: {cal_auc:.3f}")

## ROC curve on the cal set

In [ ]:
fpr, tpr, _ = roc_curve(y_cal, cal_proba)
plt.figure(figsize=(5, 5))
plt.plot(fpr, tpr, label=f"{CONFIG['model_family']} (AUC = {cal_auc:.3f})", linewidth=2)
plt.plot([0, 1], [0, 1], "--", color="grey", label="chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"ROC — {CONFIG['target']} on cal set")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Save the model

Saves to ``data/processed/student/student_model.joblib``.
Notebooks 02-05 will load this file. If you re-run this
notebook with a different model, the next notebooks
automatically pick it up.

In [ ]:
bundle = {
    "config":          CONFIG,
    "feature_columns": feat_cols,
    "model":           model,
    "cv_auc_mean":     float(cv_auc_mean),
    "cv_auc_std":      float(cv_auc_std),
    "cv_auc_ci":       [float(ci_lo), float(ci_hi)],
    "cal_auc":         float(cal_auc),
    "best_params":     str(best_params),
    "n_train":         int(train_mask.sum()),
    "n_cal":           int(cal_mask.sum()),
}
joblib.dump(bundle, STUDENT / "student_model.joblib")
print(f"Saved {STUDENT / 'student_model.joblib'}")

## Compare to your teammates

Copy your headline number into the team comparison table.
Then change ``CONFIG['model_family']`` and re-run to see
how the choice moves the needle.

### Reference numbers — bare baseline (all toggles OFF)

| model_family | CV AUC | Cal AUC |
|---|---|---|
| random_forest | 0.595 | 0.612 |
| xgboost       | 0.597 | 0.569 |
| logistic      | 0.576 | 0.573 |
| svm_rbf       | 0.571 | 0.530 |

### Reference numbers — rigorous Phase 4 (curated 92-feature
matrix, full grid search, repeated CV, Bayesian comparison)

| model_family | OOF AUC |
|---|---|
| xgboost | 0.652 |
| svm_rbf | 0.635 |

With the three toggles ON (grid search + repeated CV + class
balance) you should land somewhere in between the two
reference rows, typically lifting cal AUC to ~0.62-0.64. The
gap to the rigorous pipeline is the cost of the
simplifications: simpler feature matrix (127 vs 92 curated),
smaller hyperparameter grid, single-fold CV cell instead of
Bayesian comparison.

### Ablating the three toggles

Run with all three on, then flip one toggle to False at a
time and see which one matters most for your model family.
On random_forest you should see something like:

- all three on: cal AUC ~0.62
- turn off ``use_grid_search``: cal AUC ~0.61 (-0.01)
- turn off ``use_repeated_cv``: cal AUC unchanged (the
  headline mean barely moves; only the CI tightens)
- turn off ``use_class_balance``: cal AUC ~0.61 (-0.005)